# Financial Risk Detection — Model Training

**Purpose:** Train an XGBoost classifier on the multi-company financial dataset.

**Input:** `training_features.parquet` (from the data collection notebook)

**Outputs saved to `/kaggle/working/`:**
- `risk_model.pkl` — trained model bundle
- `model_meta.json` — feature list, class names, metrics
- `shap_values.parquet` — feature importance for the dashboard
- `feature_importance.png` — visual chart

---
> **Before running:** Upload `training_features.parquet` as a Kaggle Dataset first. Internet can be OFF.

## How to attach your parquet file

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) and click **New Dataset**
2. Upload `training_features.parquet`, name it `financial-risk-training`, click **Create**
3. Come back to this notebook, click **+ Add Data** (top right), search `financial-risk-training`, click **Add**
4. File will be at `/kaggle/input/financial-risk-training/training_features.parquet`
5. Run all cells

## Step 1 — Install & imports

In [ ]:
!pip install -q --upgrade xgboost shap
print('Packages ready')

In [ ]:
import json, os, warnings
from datetime import datetime
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
print('Imports ready')

## Step 2 — Configuration

In [ ]:
# Update the folder name if you named your dataset differently
DATA_PATH   = '/kaggle/input/financial-risk-training/training_features.parquet'
OUTPUT_DIR  = '/kaggle/working'
MODEL_PATH  = f'{OUTPUT_DIR}/risk_model.pkl'
META_PATH   = f'{OUTPUT_DIR}/model_meta.json'
SHAP_PATH   = f'{OUTPUT_DIR}/shap_values.parquet'
PLOT_PATH   = f'{OUTPUT_DIR}/feature_importance.png'

FEATURE_COLS = [
    'gross_margin', 'operating_margin', 'net_margin', 'fcf_margin',
    'roe', 'roa', 'debt_to_equity', 'current_ratio',
    'interest_coverage', 'asset_turnover', 'revenue_growth_yoy',
]
TARGET_COL   = 'risk_label'
TRAIN_CUTOFF = '2022-01-01'

print(f'Input        : {DATA_PATH}')
print(f'Output       : {OUTPUT_DIR}')
print(f'Train cutoff : {TRAIN_CUTOFF}')

## Step 3 — Load & inspect data

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f'Rows      : {len(df):,}')
print(f'Companies : {df["ticker"].nunique()}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print('\nLabel distribution:')
dist = df[TARGET_COL].value_counts()
total = len(df)
for label, count in dist.items():
    bar = chr(9608) * int(count / total * 40)
    print(f'  {label:<15} {count:>5} rows  {bar}')

df[FEATURE_COLS].describe().round(3)

## Step 4 — Time-based train / test split

We split by **date**, not randomly. A random split leaks future data into training which inflates accuracy.

**Train:** everything before 2022 | **Test:** everything from 2022 onward

In [ ]:
cutoff   = pd.Timestamp(TRAIN_CUTOFF)
train_df = df[df['date'] <  cutoff].copy()
test_df  = df[df['date'] >= cutoff].copy()

print(f'Train : {len(train_df):,} rows  ({train_df["date"].min().date()} to {train_df["date"].max().date()})')
print(f'Test  : {len(test_df):,}  rows  ({test_df["date"].min().date()} to {test_df["date"].max().date()})')
print('\nTrain labels:')
print(train_df[TARGET_COL].value_counts().to_string())
print('\nTest labels:')
print(test_df[TARGET_COL].value_counts().to_string())

## Step 5 — Prepare feature matrices

In [ ]:
def prepare_xy(df, le=None, fit=False):
    subset = df[FEATURE_COLS + [TARGET_COL]].dropna().copy()
    X      = subset[FEATURE_COLS].values.astype(np.float32)
    y_raw  = subset[TARGET_COL].astype(str).values
    if fit:
        le = LabelEncoder()
        y  = le.fit_transform(y_raw)
    else:
        y  = le.transform(y_raw)
    return X, y, le

X_train, y_train, le = prepare_xy(train_df, fit=True)
X_test,  y_test,  _  = prepare_xy(test_df,  le=le, fit=False)
n_classes = len(le.classes_)

print(f'Classes ({n_classes}): {list(le.classes_)}')
print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')

## Step 6 — Train XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, objective='multi:softprob', num_class=n_classes,
    tree_method='hist', eval_metric='mlogloss',
    early_stopping_rounds=20, random_state=42, n_jobs=-1, verbosity=0,
)
sample_weights = compute_sample_weight('balanced', y_train)
model.fit(X_train, y_train, sample_weight=sample_weights,
          eval_set=[(X_train, y_train)], verbose=False)

print('Training complete')
print(f'Best iteration : {model.best_iteration}')
print(f'Trees built    : {model.best_iteration + 1}')

## Step 7 — Evaluate on test set

Key things to check:
- **Recall on high_risk** — missing a high-risk company is the most costly error
- **Weighted F1** — overall balance across all 3 classes
- **AUC-ROC** — above 0.75 is solid for financial tabular data

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)
labels  = le.classes_

print('Classification report:')
print(classification_report(y_test, y_pred, target_names=labels))

print('Confusion matrix:')
cm    = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=labels, columns=[f'pred_{l}' for l in labels])
display(cm_df)

f1 = f1_score(y_test, y_pred, average='weighted')
try:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
except Exception:
    auc = None

print(f'\nWeighted F1  : {f1:.4f}')
if auc: print(f'Weighted AUC : {auc:.4f}')

metrics = {
    'weighted_f1':  round(f1,  4),
    'weighted_auc': round(auc, 4) if auc else None,
    'test_rows':    int(len(y_test)),
    'train_cutoff': TRAIN_CUTOFF,
    'evaluated_at': datetime.now().isoformat(),
}

## Step 8 — SHAP feature importance
SHAP explains **which ratios drove each prediction** — used by the dashboard explanation panel.

In [ ]:
print('Computing SHAP values (~30 seconds)...')
sample_size = min(500, len(X_train))
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train[:sample_size])

if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0)
else:
    mean_abs = np.abs(shap_values)

shap_df = pd.DataFrame({
    'feature':       FEATURE_COLS,
    'mean_abs_shap': mean_abs.mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('\nTop risk drivers:')
print(shap_df.to_string(index=False))

## Step 9 — Feature importance chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(shap_df['feature'][::-1], shap_df['mean_abs_shap'][::-1], color='#378ADD', edgecolor='none')
ax.set_xlabel('Mean absolute SHAP value')
ax.set_title('Feature importance — financial risk model')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150)
plt.show()
print(f'Chart saved to {PLOT_PATH}')

## Step 10 — Save all output files

In [ ]:
joblib.dump({'model': model, 'label_encoder': le}, MODEL_PATH)
print(f'Model saved     -> {MODEL_PATH}')

meta = {
    'feature_cols': FEATURE_COLS,
    'classes':      list(le.classes_),
    'metrics':      metrics,
    'trained_at':   datetime.now().isoformat(),
    'model_type':   'XGBClassifier',
    'train_cutoff': TRAIN_CUTOFF,
}
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Meta saved      -> {META_PATH}')

shap_df.to_parquet(SHAP_PATH, index=False)
print(f'SHAP saved      -> {SHAP_PATH}')

print('\nOutput files:')
for fname in ['risk_model.pkl', 'model_meta.json', 'shap_values.parquet', 'feature_importance.png']:
    fpath = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        print(f'  {fname:<35} {size/1024:.1f} KB')

## Done!

Download these 4 files from the **Output tab** (right panel) and place them in `models/` in your project:

```
financial-risk-app/
  models/
    risk_model.pkl
    model_meta.json
    shap_values.parquet
    feature_importance.png
```

**Next step:** Run the EPAM single-company pipeline (Step 4) to pull live EPAM data for the dashboard.